# SDSS re-import



In [3]:
from dask.distributed import Client
from hats_import.pipeline import pipeline_with_client, pipeline
from hats_import import ImportArguments, CollectionArguments, VerificationArguments
import hats_import

hats_import.__version__

'0.9.0'

In [2]:
args = ImportArguments.reimport_from_hats(
    "/epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_spectra/",
    "/epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_collection/",
    output_artifact_name="sdss_dr7_spectra",
    
    ra_column='RA',
    dec_column='DEC',
    expected_total_rows=1_640_953,
    pixel_threshold= 15_000,
    highest_healpix_order=8,
    skymap_alt_orders=[2, 4, 6],
    row_group_kwargs={"num_rows": 3_000},

    completion_email_address="delucchi@andrew.cmu.edu",
    progress_bar=True,
    simple_progress_bar=True,
)

with Client(local_directory="/epyc/data3/hats/tmp/", n_workers=10, threads_per_worker=1) as client:
    pipeline_with_client(args, client)

Validating catalog at path /epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_spectra ... 
Found 1304 partitions.
Approximate coverage is 45.19 % of the sky.


Catalog: Finishing : 100%|██████████| 6/6 [00:04<00:00,  1.30it/s]


In [4]:
args = (
    CollectionArguments(
        output_artifact_name="sdss_dr7_collection",
        output_path="/epyc/data3/hats/catalogs/sdss_dr7/",
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(
        output_artifact_name="sdss_dr7_spectra",
    )
    .add_margin(margin_threshold=5.0, is_default=True)
)

with Client(local_directory="/epyc/data3/hats/tmp/", n_workers=10, threads_per_worker=1) as client:
    pipeline_with_client(args, client)

Margin: Finishing :  25%|██▌       | 1/4 [00:00<00:01,  2.99it/s]/astro/store/shire/mmd11/conda_envs/hatsv09/lib/python3.13/site-packages/hats/catalog/partition_info.py:113: UserWarning: Computing partitions from catalog parquet files. This may be slow.
  warnings.warn("Computing partitions from catalog parquet files. This may be slow.")
Collection: Finishing : 100%|██████████| 2/2 [00:00<00:00, 36.49it/s]


In [6]:
args = VerificationArguments(
    input_catalog_path="/epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_collection",
    output_path="./verification/sdss_dr7",
)
pipeline(args)

Loading dataset and schema.

Starting: Test hats.io.validation.is_valid_collection.
Validating collection at path /epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_collection ... 
Validating catalog at path /epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_collection/sdss_dr7_spectra ... 
Found 231 partitions.
Approximate coverage is 64.11 % of the sky.
Validating catalog at path /epyc/data3/hats/catalogs/sdss_dr7/sdss_dr7_collection/sdss_dr7_spectra_5arcs ... 
Found 226 partitions.
Approximate coverage is 58.77 % of the sky.
Result: PASSED

Starting: Test that files in _metadata match the data files on disk.
Result: PASSED

Starting: Test that number of rows are equal.
	file footers vs catalog properties
	file footers vs _metadata
Result: PASSED

Starting: Test that schemas are equal, excluding metadata.
	_common_metadata vs truth
	_metadata vs truth
	file footers vs truth
Result: PASSED

Verifier results written to verification/sdss_dr7/verifier_results.csv
Elapsed time (seconds): 0.34
